In [ ]:
import numpy as np
import hcipy as hp
import matplotlib.pyplot as plt

from utils.magaox import make_magaox_bump_mask
from utils.apodized_mask_generator import make_magaox_piaa_lyot_stop
from utils.PIAACMC import (
    load_fpm_csv, center_fpm_map, make_fpm_apodizer, make_measured_fpm_apodizer,
    DESIGN_RADII, DESIGN_HEIGHTS, F_NUMBER, SPATIAL_RESOLUTION,
)


Define constants, 

INIT pupil and aperture for Magellan, may change to MagAO-X

Supersample the pupil by factor 8 for more detail

In [ ]:
# Telescope / instrument constants
aperture_diameter = 6.5            # m
effective_focal_length = 71.5   # m

pupil_grid = hp.make_pupil_grid(500, diameter=aperture_diameter)

magellan_pupil_generator = make_magaox_bump_mask()
telescope_pupil = hp.evaluate_supersampled(magellan_pupil_generator, pupil_grid, 8)

hp.imshow_field(telescope_pupil, cmap='gray')
plt.colorbar()
plt.xlabel('x [m]')
plt.ylabel('y [m]')
plt.show()


Create and select the filters: 

- Narrowband 875 filter $\lambda$ = 875nm, bandwidth = 3\%
- Broadband z' filter (here z): $\lambda$ = 908nm, bandwidth = 14\%

In [ ]:
# Define and select filter
filter_875 = {'name': '875 nm', 'wavelength': 875e-9, 'bandwidth': 0.03} 
filter_z   = {'name': 'z band', 'wavelength': 908e-9, 'bandwidth': 0.13/0.908}
# 875 filter is to narrowband, not enough flux for the detection of faint objects. Use z band instead.
# filter = filter_875
filter = filter_z
wavelength = filter['wavelength']
bandwidth = filter['bandwidth']


Propagate a Monochomatic (at central wavelength of the filter) wavefront towards the focal plane.\
Propagate a bandwidth for each filter

In [ ]:
wavelength = filter['wavelength']
bandwidth = filter['bandwidth']
wavefront = hp.Wavefront(telescope_pupil, wavelength=wavelength) # create wavefront object

focal_grid = hp.make_focal_grid(q=8, num_airy=16, pupil_diameter=aperture_diameter,
                                focal_length=effective_focal_length, reference_wavelength=wavelength)

fraun = hp.FraunhoferPropagator(pupil_grid, focal_grid, focal_length=effective_focal_length)

focal_image = fraun.forward(wavefront) #plot the monochromatic PSF

focal_total = 0
for wlen in wavelength * np.linspace(1 - bandwidth / 2., 1 + bandwidth / 2., 11):
    wavefront = hp.Wavefront(telescope_pupil, wlen)
    focal_total += fraun.forward(wavefront).power

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].set_title('Pupil Plane')
ax[1].set_title('Focal Plane')
ax[2].set_title('Focal Plane (Broadband)')

hp.imshow_field(wavefront.amplitude, grid_units = aperture_diameter, ax=ax[0], cmap='inferno')
ax[0].set_xlabel(r'Pupil plane distance [$m$]')
ax[0].set_ylabel(r'Pupil plane distance [$m$]')
ax[0].set_title(filter['name'] + ' — pupil plane')
ax[0].set_aspect('equal')
plt.colorbar(ax[0].images[0], ax=ax[0], label='Amplitude')

hp.imshow_field(np.log10(focal_image.power / focal_image.power.max()), grid_units=1e-6, ax=ax[1],cmap='inferno')
ax[1].set_xlabel(r'Focal plane distance [$\mu m$]')
ax[1].set_ylabel(r'Focal plane distance [$\mu m$]')
ax[1].set_title(filter['name'] + ' — monochromatic PSF')
ax[1].set_aspect('equal')
plt.colorbar(ax[1].images[0], ax=ax[1], label='Log10(power)')

hp.imshow_field(np.log10(focal_total / focal_total.max()), grid_units=1e-6, ax=ax[2],cmap='inferno')
ax[2].set_xlabel(r'Focal plane distance [$\mu m$]')
ax[2].set_ylabel(r'Focal plane distance [$\mu m$]')
ax[2].set_title(filter['name'] + ' — broadband PSF')
ax[2].set_aspect('equal')

plt.colorbar(ax[2].images[0], ax=ax[2], label='Log10(power)')
plt.show()


Import CSV for PIAA and FPM

In [ ]:
piaa0_csv = np.loadtxt('utils/piaa0_CaF2.csv', delimiter=',', skiprows=1)   # [radius_m, cut_depth_m]
piaa1_csv = np.loadtxt('utils/piaa1_CaF2.csv', delimiter=',', skiprows=1)
radial_d0, profile0 = piaa0_csv[:, 0], -piaa0_csv[:, 1]
radial_d1, profile1 = piaa1_csv[:, 0], -piaa1_csv[:, 1]
profile0 = profile0 - np.min(radial_d0)
profile1 = profile1 - np.min(radial_d1)

# Load and centre the measured FPM height map
fpm_height, fpm_pixel_size = load_fpm_csv('utils/FPM.csv')
fpm_height = center_fpm_map(fpm_height, fpm_pixel_size)
print('FPM map:', fpm_height.shape, '  pixel size [m]:', fpm_pixel_size)


Plot lense profile and FPM shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(radial_d0 * 1e3, profile0, label='PIAA0', color='purple')
ax[0].plot(radial_d1 * 1e3, profile1, label='PIAA1', color='hotpink')
ax[0].set_xlabel('Radius [mm]')
ax[0].set_ylabel('height [µm]')
ax[0].set_title('Measured PIAA lens surfaces')
ax[0].legend()

ax[1].imshow(fpm_height, cmap='inferno')
ax[1].set_xlabel('Pixel index')
ax[1].set_ylabel('Pixel index')
ax[1].set_title('Measured FPM height map (centred)')
plt.colorbar(ax[1].images[0], ax=ax[1], label='Height [µm]')
plt.show()


Create the lense propagators 

In [ ]:
d_pupil_piaa0 = 37.2e-3   # m, pupil plane -> PIAA0
d_piaa0_piaa1 = 60.2e-3   # m, PIAA0 -> PIAA1
pupil_diameter = 8.6e-3   # m, pupil diameter (for the wavefront)
ref_ind_lenses = hp.get_refractive_index('CAF2')   # n(lambda)

piaa_pupil_grid = hp.make_pupil_grid(500, diameter=1.3*pupil_diameter)
aperture = hp.evaluate_supersampled(make_magaox_bump_mask(aperture_scale="pupil"), piaa_pupil_grid, 8)

prop_pupil_piaa0 = hp.FresnelPropagator(aperture.grid,d_pupil_piaa0) #AngularSpectrumPropagator
prop_piaa0_piaa1 = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1)
prop_piaa1_pupil = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1+d_pupil_piaa0)

prop_LS_piaa1 = hp.FresnelPropagator(aperture.grid,d_pupil_piaa0)
prop_piaa1_piaa0 = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1)
prop_piaa0_LS = hp.FresnelPropagator(aperture.grid,d_piaa0_piaa1+d_pupil_piaa0)


sag0 = hp.Field(np.interp(aperture.grid.as_('polar').r,radial_d0, profile0),aperture.grid)
sag1 = hp.Field(np.interp(aperture.grid.as_('polar').r,radial_d1, profile1),aperture.grid)

piaa0 = hp.SurfaceApodizer(sag0, ref_ind_lenses)
piaa1 = hp.SurfaceApodizer(sag1, ref_ind_lenses)

separation = 0
wf_pupil = hp.Wavefront(aperture*np.exp(2j * np.pi * separation * aperture.grid.x / pupil_diameter))

wf_pupil.wavelength = wavelength
wf_pupil.total_power = 1.0
wf_piaa0 = piaa0.forward(prop_pupil_piaa0.forward(wf_pupil))
wf_piaa1 = piaa1.forward(prop_piaa0_piaa1.forward(wf_piaa0))
wf_apodized = prop_piaa1_pupil.backward(wf_piaa1)


#plot lenses
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].set_title('PIAA0 sag [m]')
ax[1].set_title('PIAA1 sag [m]')
ax[2].set_title('Apodized pupil power')
hp.imshow_field(piaa0.surface_sag, cmap='gray', grid_units=1e-3, ax=ax[0])
hp.imshow_field(piaa1.surface_sag, cmap='gray', grid_units=1e-3, ax=ax[1])
hp.imshow_field(wf_apodized.power, cmap='gray', grid_units=1e-3, ax=ax[2])
ax[0].set_xlabel('x [m]')
ax[0].set_ylabel('y [m]')
ax[1].set_xlabel('x [m]')
ax[1].set_ylabel('y [m]')
ax[2].set_xlabel('x [m]')
ax[2].set_ylabel('y [m]')
plt.colorbar(ax[0].images[0], ax=ax[0], label='Sag [m]')
plt.colorbar(ax[1].images[0], ax=ax[1], label='Sag [m]')
plt.colorbar(ax[2].images[0], ax=ax[2], label='Power')
plt.show()


In [ ]:
# Focal-plane scale and FPM radius (papers 2 & 3)
spatial_resolution = SPATIAL_RESOLUTION          # 1 lambda/D (~62.6 µm)
fpm_radius = 0.75 * spatial_resolution           # FPM radius = 0.75 lambda/D (~47 µm)
print(f'Focal-plane scale (1 lambda/D) = {spatial_resolution * 1e6:.2f} µm,  FPM radius = {fpm_radius * 1e6:.2f} µm')

# Focal grid covering the FPM, and the propagator that reaches it
cor_grid = hp.make_focal_grid(q=50, num_airy=20, spatial_resolution=spatial_resolution)

n_mask = hp.get_refractive_index('SILICA')       # FPM substrate = fused silica
prop_piaa1_focal = hp.FraunhoferPropagator(piaa_pupil_grid, cor_grid, focal_length=F_NUMBER * pupil_diameter)

# Focal-plane image right before the FPM (apodized pupil, no mask yet)
wavefront_focal = prop_piaa1_focal.forward(wf_apodized)

plt.figure(figsize=(6, 5))
hp.imshow_field(np.log10(wavefront_focal.power / wavefront_focal.power.max()), grid_units=1e-6, cmap='inferno')
plt.xlabel(r'Focal plane distance [$\mu m$]')
plt.ylabel(r'Focal plane distance [$\mu m$]')
plt.title('Focal plane after PIAA lenses (before FPM)')
plt.colorbar(label='Log10(power)')
plt.show()


In [ ]:
# Design FPM: 10 concentric rings from the module design constants
FPM_apodizer = make_fpm_apodizer(DESIGN_RADII, DESIGN_HEIGHTS, cor_grid, n_mask)

# Apply the FPM at the f/69 focal plane
wf_fpm = FPM_apodizer.forward(prop_piaa1_focal.forward(wf_apodized))


In [ ]:

# Quick check of the design FPM surface
hp.imshow_field(FPM_apodizer.surface_sag * 1e6, cmap='inferno', grid_units=1e-6)
plt.colorbar(label='FPM surface height [um]')
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.title('FPM surface (design)')
plt.show()


Implement FPM using the csv data


In [ ]:
# Measured FPM: sample the centred height map onto the focal grid
fpm_apodizer = make_measured_fpm_apodizer(fpm_height, fpm_pixel_size, cor_grid, n_mask)
wf_fpm = fpm_apodizer.forward(prop_piaa1_focal.forward(wf_apodized))


In [ ]:

# Quick check of the measured FPM surface
hp.imshow_field(fpm_apodizer.surface_sag * 1e6, cmap='inferno', grid_units=1e-6)
plt.colorbar(label='FPM surface height [µm]')
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.title('FPM surface (measured)')
plt.show()


Propagate through the Lyot stop

In [ ]:
# Propagate the FPM phase-changed wavefront to the Lyot pupil plane.
# On-axis light, phase-shifted by the FPM, interferes destructively inside the
# geometric pupil and is diffracted outside it.
prop_fpm_lyot = hp.FraunhoferPropagator(cor_grid, piaa_pupil_grid, focal_length=F_NUMBER * pupil_diameter)
wf_lyot = prop_fpm_lyot.forward(wf_fpm)

# Lyot stop: block the light diffracted outside the apodized pupil
lyot_stop = hp.evaluate_supersampled(make_magaox_piaa_lyot_stop(), piaa_pupil_grid, 8)
wf_lyot_stopped = hp.Wavefront(wf_lyot.electric_field * lyot_stop, wavelength=wavelength)

log_lyot = np.log10(np.maximum(wf_lyot.power / wf_lyot.power.max(), 1e-15))
log_lyot_stopped = np.log10(np.maximum(wf_lyot_stopped.power / wf_lyot_stopped.power.max(), 1e-15))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
hp.imshow_field(log_lyot_stopped, cmap='inferno', ax=ax[0])
ax[0].set_title('Lyot pupil after Lyot stop (central light removed)')
ax[0].set_aspect('equal')
hp.imshow_field(log_lyot, cmap='inferno', ax=ax[1])
ax[1].set_title('Lyot pupil after FPM (interference pattern)')
ax[1].set_aspect('equal')
plt.show()


Propagate through the inverse PIAA lenses


In [ ]:
# Inverse PIAA lenses: de-apodize the Lyot-stopped pupil back to a round pupil
wavefront_pupil_out = prop_piaa0_LS.forward(piaa0.backward(
                        prop_piaa1_piaa0.forward(piaa1.backward(
                        prop_LS_piaa1.forward(wf_lyot_stopped)))))

# Propagate to the detector focal plane: the final coronagraphic PSF
detector_grid = hp.make_focal_grid(q=4, num_airy=20, pupil_diameter=pupil_diameter,
                                   focal_length=F_NUMBER * pupil_diameter, reference_wavelength=wavelength)

prop_detector = hp.FraunhoferPropagator(aperture.grid, detector_grid, focal_length=F_NUMBER * pupil_diameter)
psf = prop_detector.forward(wavefront_pupil_out)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
hp.imshow_field(wavefront_pupil_out.power / wavefront_pupil_out.power.max(), cmap='inferno', ax=ax[0])
ax[0].set_title('De-apodized pupil (after inverse PIAA)')
ax[0].set_aspect('equal')
hp.imshow_field(np.log10(np.maximum(psf.power / psf.power.max(), 1e-15)), grid_units=1e-6, cmap='inferno', ax=ax[1])
ax[1].set_title('Final coronagraphic PSF')
ax[1].set_xlabel(r'Detector plane [$\mu m$]')
ax[1].set_ylabel(r'Detector plane [$\mu m$]')
plt.show()
